# 개별종목 조합C — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합C 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합C의 피처 값만 지정합니다.
import json

COMBINATION = 'C'
FEATURE_COLUMNS = (
    'ret_1',
    'ret_5',
    'overnight_gap',
    'intraday_return',
    'close_location',
    'bb_position',
    'rsi_14',
    'volume_z_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 24개 노트북이 각각 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20100331 ~ 20240822
학습 행·종목: 172939 163
조합C 피처: ('ret_1', 'ret_5', 'overnight_gap', 'intraday_return', 'close_location', 'bb_position', 'rsi_14', 'volume_z_20')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20130410,20130705,0.3399,0.3701,-0.0302,0.3239,0.3853,0.3478
1,2,balanced,999,20140414,20140711,0.2961,0.4741,-0.1780,0.2894,0.4805,0.3366
2,3,balanced,1248,20150421,20150716,0.3435,0.3300,0.0135,0.3235,0.4452,0.3637
3,4,balanced,1496,20160422,20160719,0.3368,0.4108,-0.0740,0.3142,0.5080,0.3695
4,5,balanced,1745,20170424,20170721,0.3057,0.4178,-0.1121,0.2916,0.5574,0.3532
5,6,balanced,1994,20180503,20180731,0.3390,0.3907,-0.0517,0.3208,0.4340,0.3584
6,7,balanced,2243,20190514,20190806,0.2857,0.4612,-0.1755,0.2828,0.3050,0.2908
7,8,balanced,2492,20200518,20200807,0.3722,0.3146,0.0576,0.3520,0.5011,0.3988
8,9,balanced,2741,20210518,20210810,0.3538,0.4421,-0.0884,0.3539,0.3812,0.3625
9,10,balanced,2989,20220519,20220812,0.3420,0.3343,0.0077,0.3414,0.3049,0.3285


,OOS 폴드 평균
accuracy,0.3331
training_majority_baseline_accuracy,0.3846
accuracy_minus_training_majority_baseline,-0.0515
macro_f1,0.3222
down_recall,0.4208
core_harmonic_mean,0.3507


재실행 명령: python scripts/run_stock_model_experiment.py
